# Agentes con Memoria

Implementamos los tres tipos de memoria: episódica (JSON), factual (diccionario) y semántica (embeddings con ChromaDB).

In [ ]:
import anthropic
import json
import os
from datetime import datetime
from pathlib import Path

client = anthropic.Anthropic()

DIRECTORIO_MEMORIA = Path("memoria_agente_demo")
DIRECTORIO_MEMORIA.mkdir(exist_ok=True)

print("Directorio de memoria:", DIRECTORIO_MEMORIA.absolute())

## 1. Memoria episódica: guardar conversaciones

In [ ]:
def guardar_sesion(user_id: str, mensajes: list, resumen: str = "") -> str:
    """Guarda los mensajes de una sesión en disco."""
    sesion = {
        "user_id": user_id,
        "fecha": datetime.now().isoformat(),
        "resumen": resumen,
        "num_mensajes": len(mensajes),
        "mensajes": [
            {"role": m["role"], "content": m["content"][:500]}
            for m in mensajes
            if isinstance(m.get("content"), str)
        ]
    }
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    ruta = DIRECTORIO_MEMORIA / f"{user_id}_{ts}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(sesion, f, ensure_ascii=False, indent=2)
    return str(ruta)

def cargar_sesiones_recientes(user_id: str, n: int = 3) -> list:
    """Carga las N sesiones más recientes del usuario."""
    archivos = sorted(
        DIRECTORIO_MEMORIA.glob(f"{user_id}_*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )[:n]
    sesiones = []
    for archivo in archivos:
        with open(archivo, encoding="utf-8") as f:
            sesiones.append(json.load(f))
    return sesiones

def resumir_sesion(mensajes: list) -> str:
    """Usa Claude para resumir la sesión."""
    texto = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in mensajes
        if isinstance(m.get("content"), str)
    )
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{
            "role": "user",
            "content": f"Resume en 2-3 líneas los puntos clave de esta conversación:\n{texto[:2000]}"
        }]
    )
    return resp.content[0].text


# Demostración: guardar una sesión de ejemplo
mensajes_ejemplo = [
    {"role": "user", "content": "Hola, soy Ana García, directora de operaciones en LogiTech SA"},
    {"role": "assistant", "content": "Hola Ana, encantado de conocerte. ¿En qué puedo ayudarte hoy?"},
    {"role": "user", "content": "Necesito automatizar los informes semanales de mi equipo"},
    {"role": "assistant", "content": "Entiendo. Para los informes semanales podemos usar Python con pandas y enviarlos por email automáticamente."}
]

resumen = resumir_sesion(mensajes_ejemplo)
ruta = guardar_sesion("usuario_001", mensajes_ejemplo, resumen)
print(f"Sesión guardada en: {ruta}")
print(f"Resumen: {resumen}")

## 2. Memoria factual: perfil del usuario

In [ ]:
class MemoriaFactual:
    """Almacén de hechos y preferencias del usuario."""

    def __init__(self, user_id: str):
        self.user_id = user_id
        self.ruta = DIRECTORIO_MEMORIA / f"{user_id}_perfil.json"
        self._datos = self._cargar()

    def _cargar(self) -> dict:
        if self.ruta.exists():
            with open(self.ruta, encoding="utf-8") as f:
                return json.load(f)
        return {"preferencias": {}, "datos": {}, "historial_temas": []}

    def _guardar(self):
        with open(self.ruta, "w", encoding="utf-8") as f:
            json.dump(self._datos, f, ensure_ascii=False, indent=2)

    def actualizar(self, clave: str, valor, categoria: str = "datos"):
        self._datos.setdefault(categoria, {})[clave] = valor
        self._guardar()
        print(f"  💾 Memorizado: {clave} = {valor}")

    def obtener(self, clave: str, categoria: str = "datos", default=None):
        return self._datos.get(categoria, {}).get(clave, default)

    def agregar_tema(self, tema: str):
        self._datos["historial_temas"].append({
            "tema": tema, "fecha": datetime.now().isoformat()
        })
        self._guardar()

    def resumen_perfil(self) -> str:
        datos = self._datos.get("datos", {})
        prefs = self._datos.get("preferencias", {})
        temas = [t["tema"] for t in self._datos.get("historial_temas", [])[-5:]]
        return (
            f"Datos: {datos}\n"
            f"Preferencias: {prefs}\n"
            f"Temas recientes: {', '.join(temas) if temas else 'ninguno'}"
        )


# Demo de memoria factual
memoria = MemoriaFactual("usuario_001")
memoria.actualizar("nombre", "Ana García")
memoria.actualizar("empresa", "LogiTech SA")
memoria.actualizar("cargo", "Directora de Operaciones")
memoria.actualizar("lenguaje_preferido", "Python", categoria="preferencias")
memoria.agregar_tema("automatización de informes")
memoria.agregar_tema("gestión de equipos")

print("\nPerfil del usuario:")
print(memoria.resumen_perfil())

## 3. Agente con memoria completa

In [ ]:
class AgenteConMemoria:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.historial_sesion = []
        self.memoria_factual = MemoriaFactual(user_id)

    def _construir_system(self) -> str:
        perfil = self.memoria_factual.resumen_perfil()
        sesiones_previas = cargar_sesiones_recientes(self.user_id, n=2)
        contexto_previo = ""
        if sesiones_previas:
            contexto_previo = "\n".join(
                f"- {s['fecha'][:10]}: {s.get('resumen', 'sin resumen')}"
                for s in sesiones_previas
            )

        return f"""Eres un asistente personal que recuerda al usuario entre sesiones.

PERFIL DEL USUARIO:
{perfil}

SESIONES ANTERIORES:
{contexto_previo if contexto_previo else 'Sin sesiones previas'}

INSTRUCCIONES:
- Usa el perfil para personalizar tus respuestas
- Si aprendes algo nuevo del usuario, asúmelo para esta sesión
- Sé consistente con lo que sabes del usuario"""

    def _extraer_y_guardar_hechos(self, mensaje_usuario: str, respuesta: str):
        """Claude extrae hechos del usuario y los guarda."""
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            messages=[{
                "role": "user",
                "content": f"""Del siguiente intercambio, extrae datos del usuario para recordar.
Si no hay datos nuevos, responde con {{}}.
Responde SOLO con JSON: {{"nombre": "...", "empresa": "...", "cargo": "...", "preferencia": "..."}}
Solo incluye campos que aparezcan explícitamente.

Usuario dijo: {mensaje_usuario}
Asistente respondió: {respuesta[:200]}"""
            }]
        ).content[0].text.strip()

        try:
            if "```" in resp:
                resp = resp.split("```")[1].lstrip("json")
            hechos = json.loads(resp)
            for clave, valor in hechos.items():
                if valor:
                    self.memoria_factual.actualizar(clave, valor)
        except (json.JSONDecodeError, KeyError):
            pass

    def chat(self, mensaje: str) -> str:
        self.historial_sesion.append({"role": "user", "content": mensaje})

        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=500,
            system=self._construir_system(),
            messages=self.historial_sesion
        )

        respuesta = resp.content[0].text
        self.historial_sesion.append({"role": "assistant", "content": respuesta})
        self._extraer_y_guardar_hechos(mensaje, respuesta)
        return respuesta

    def finalizar_sesion(self):
        if self.historial_sesion:
            resumen = resumir_sesion(self.historial_sesion)
            ruta = guardar_sesion(self.user_id, self.historial_sesion, resumen)
            print(f"\n💾 Sesión guardada: {ruta}")
            print(f"📝 Resumen: {resumen}")


# Demostración: sesión con un nuevo usuario
print("=" * 60)
print("SESIÓN 1: Primer contacto con usuario_002")
print("=" * 60)

agente = AgenteConMemoria("usuario_002")

intercambios = [
    "Hola, soy Carlos López, CTO de FinTech Solutions",
    "Estoy buscando integrar IA en nuestro sistema de detección de fraude",
    "¿Recuerdas mi nombre y en qué empresa trabajo?"
]

for mensaje in intercambios:
    print(f"\n👤 Usuario: {mensaje}")
    resp = agente.chat(mensaje)
    print(f"🤖 Agente: {resp}")

agente.finalizar_sesion()

## 4. Segunda sesión: el agente recuerda

In [ ]:
print("=" * 60)
print("SESIÓN 2: El agente recuerda al usuario")
print("=" * 60)

agente2 = AgenteConMemoria("usuario_002")

mensaje = "Hola de nuevo, ¿recuerdas de qué hablamos la última vez?"
print(f"\n👤 Usuario: {mensaje}")
resp = agente2.chat(mensaje)
print(f"🤖 Agente: {resp}")

mensaje2 = "¿Qué sabes sobre mí?"
print(f"\n👤 Usuario: {mensaje2}")
resp2 = agente2.chat(mensaje2)
print(f"🤖 Agente: {resp2}")

agente2.finalizar_sesion()

## 5. Compresión de contexto para sesiones largas

In [ ]:
def comprimir_historial(historial: list, max_tokens: int = 3000) -> list:
    """Comprime el historial cuando supera el límite de tokens."""
    tokens_est = sum(len(str(m.get("content", ""))) // 4 for m in historial)

    if tokens_est <= max_tokens:
        print(f"  ✓ Historial en límite ({tokens_est} tokens estimados)")
        return historial

    print(f"  ⚠️ Comprimiendo historial ({tokens_est} tokens → objetivo {max_tokens})")

    turnos_recientes = historial[-8:]
    historial_antiguo = historial[:-8]

    texto_antiguo = "\n".join(
        f"{m['role']}: {m['content']}"
        for m in historial_antiguo
        if isinstance(m.get("content"), str)
    )

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"Resume en 3-4 puntos clave esta conversación:\n{texto_antiguo[:2000]}"
        }]
    )
    resumen = resp.content[0].text

    return [
        {"role": "user", "content": f"[CONTEXTO PREVIO RESUMIDO]: {resumen}"},
        {"role": "assistant", "content": "Entendido. Continuemos desde donde estábamos."}
    ] + turnos_recientes


# Simular historial largo
historial_largo = []
for i in range(20):
    historial_largo.append({"role": "user", "content": f"Pregunta {i+1}: ¿Cómo implemento la feature {i+1} en Python?"})
    historial_largo.append({"role": "assistant", "content": f"Para la feature {i+1}, necesitas usar la clase correspondiente y configurar los parámetros adecuados. Aquí tienes un ejemplo detallado de implementación."})

print(f"Historial original: {len(historial_largo)} mensajes")
historial_comprimido = comprimir_historial(historial_largo, max_tokens=1000)
print(f"Historial comprimido: {len(historial_comprimido)} mensajes")
print("\nPrimer mensaje (resumen del contexto anterior):")
print(historial_comprimido[0]["content"][:300])

## 6. Limpieza de archivos de demo

In [ ]:
# Ver archivos creados
archivos = list(DIRECTORIO_MEMORIA.glob("*"))
print(f"Archivos de memoria creados ({len(archivos)}):")
for a in archivos:
    tamaño = a.stat().st_size
    print(f"  {a.name} ({tamaño} bytes)")

# Descomenta para limpiar:
# import shutil
# shutil.rmtree(DIRECTORIO_MEMORIA)
# print("Directorio de memoria eliminado")